# 第72章 电商销售数据分析

使用 Gapminder 真实面板数据（142国 × 12期 = 1704行），完成收敛性分析、人口加权增量分解与冲击事件定位，学习诊断型分析范式。

## 项目背景

世界卫生统计年报要回答一个问题：1952 到 2007 年间各洲预期寿命普遍上升，但国家之间的差距是缩小了还是扩大了？上升究竟来自各国自身改善，还是人口权重变化带来的算术效应？哪些国家出现过倒退，倒退发生在哪一年？本项目使用 Gapminder 基金会公开数据（随 plotly 离线分发）完成这份诊断。

## 学习目标

- 读取真实公开数据集并核验面板结构的完整性
- 用离散度指标判断国家间差距是收敛还是扩大
- 用 shift-share 方法把大洲变化分解为国内改善与人口结构两部分
- 用 IQR 与逐期差分定位异常国家和冲击发生的年份
- 区分相关关系的强度与函数形式，避免把相关写成因果


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| country | 国家名称 | 面板个体维度，142个 |
| continent | 所属大洲 | Asia/Europe/Africa/Americas/Oceania |
| year | 观测年份 | 1952-2007，每5年一期，共12期 |
| lifeExp | 预期寿命（岁） | 核心结果指标 |
| pop | 总人口（人） | 用于人口加权与结构分解 |
| gdpPercap | 人均GDP（国际元） | 解释变量，购买力平价计价 |
| iso_alpha | ISO三位国家码 | 用于关联外部数据 |

## 数据质量检查清单

- （country, year）组合是否唯一，可否作为面板主键
- 是否为平衡面板：每个国家的观测期数是否一致
- 年份间隔是否等距，能否直接做逐期差分
- lifeExp / pop / gdpPercap 是否均为正值
- 缺失值分布是否集中在特定国家或年份


## 项目任务

1. 加载 Gapminder 数据并记录来源与字段口径
2. 完成面板结构审计与取值范围检查
3. 对比首末年份的分布形态，识别分布是否变窄
4. 计算标准差、变异系数与 P90-P10 差距，判定收敛方向
5. 用 shift-share 分解各洲预期寿命变化的来源
6. 定位倒退国家与单期最大跌幅发生的年份
7. 分析预期寿命与人均GDP的相关形式随时间的演变
8. 输出事实→假设→验证方案的三段式结论


## 1. 加载真实数据并记录溯源

分析的第一步不是算指标，而是说清数据从哪来、每个字段什么含义、覆盖范围多大。plotly 把 Gapminder 数据以压缩 CSV 随包分发，因此这里读取的是本地文件，不需要联网。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)

# 数据来源：Gapminder 基金会 https://www.gapminder.org/data/
# 分发方式：随 plotly 包内置的 gapminder.csv.gz，本地读取，无需网络
gap = px.data.gapminder()

print("数据形状（行, 列）:", gap.shape)
print("字段:", list(gap.columns))
print("年份范围:", gap["year"].min(), "到", gap["year"].max())
print("国家数:", gap["country"].nunique(), "  大洲数:", gap["continent"].nunique())
print()
print("前5行:")
print(gap.head())
print()
print("字段类型:")
print(gap.dtypes)


## 2. 面板结构审计：主键、平衡性与取值范围

面板数据的分析前提是结构可靠。要确认（国家, 年份）能唯一标识一行、每个国家的观测期数一致、年份等距，否则后面的逐期差分和分组对比都会算错。真实数据集也必须审计，不能因为它公开就假设它干净。


In [ ]:
# 1) 主键唯一性
dup = gap.duplicated(subset=["country", "year"]).sum()
print("重复的(country, year)组合数:", dup)

# 2) 平衡面板检查：每个国家的观测期数
periods = gap.groupby("country")["year"].nunique()
print("每国观测期数 - 最小:", periods.min(), " 最大:", periods.max())
print("是否为平衡面板:", periods.min() == periods.max())

# 3) 年份是否等距
years = np.sort(gap["year"].unique())
print("年份序列:", years.tolist())
print("年份间隔:", np.unique(np.diff(years)).tolist())

# 4) 缺失值与取值范围
print()
print("缺失值统计:")
print(gap.isna().sum()[lambda s: s > 0] if gap.isna().sum().sum() else "无缺失值")
print()
print("数值字段分布:")
print(gap[["lifeExp", "pop", "gdpPercap"]].describe().T[["min", "25%", "50%", "75%", "max"]])

# 5) 业务合理性：三个指标都应为正
for col in ["lifeExp", "pop", "gdpPercap"]:
    bad = (gap[col] <= 0).sum()
    print(f"{col} 非正值行数: {bad}")

print()
print("每洲国家数:")
print(gap.groupby("continent")["country"].nunique().sort_values(ascending=False))


## 3. 首末年份分布对比：整体上移还是差距收窄

诊断分析先看分布再看均值。均值上升有两种完全不同的形态：整个分布平移（所有国家一起改善），或者尾部追赶（落后国家改善更快、分布变窄）。二者对应的政策含义完全不同，所以必须先把分布画出来。


In [ ]:
y0, y1 = years[0], years[-1]
first = gap[gap["year"] == y0]["lifeExp"]
last = gap[gap["year"] == y1]["lifeExp"]

stat = pd.DataFrame({
    str(y0): first.describe(),
    str(y1): last.describe()
}).T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)
print("首末年份分布对比:")
print(stat)
print()
print(f"均值提升: {last.mean() - first.mean():.2f} 岁")
print(f"标准差变化: {last.std() - first.std():.2f} 岁（负值说明国家间差距收窄）")
print(f"最低值提升: {last.min() - first.min():.2f} 岁")
print(f"最高值提升: {last.max() - first.max():.2f} 岁")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
bins = np.arange(20, 90, 4)
axes[0].hist(first, bins=bins, alpha=0.65, label=str(y0), color="#94a3b8", edgecolor="white")
axes[0].hist(last, bins=bins, alpha=0.65, label=str(y1), color="#2563eb", edgecolor="white")
axes[0].set_title("Life expectancy distribution")
axes[0].set_xlabel("years")
axes[0].set_ylabel("countries")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

order = ["Africa", "Asia", "Americas", "Europe", "Oceania"]
data = [gap[(gap["year"] == y1) & (gap["continent"] == c)]["lifeExp"].values for c in order]
axes[1].boxplot(data, showmeans=True)
axes[1].set_xticklabels(order, rotation=15)
axes[1].set_title(f"By continent, {y1}")
axes[1].set_ylabel("years")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 4. 收敛性检验：离散度指标的时间趋势

上一步只看了两个时点，可能是偶然。收敛是一个过程，要看每一期的离散度是否单调下降。这里用三个互补指标：标准差（绝对差距）、变异系数（相对差距）、P90-P10 分位差（对极端值不敏感）。三者同向才能下结论。


In [ ]:
conv = gap.groupby("year")["lifeExp"].agg(
    mean="mean", std="std",
    p10=lambda s: s.quantile(0.10),
    p90=lambda s: s.quantile(0.90)
).reset_index()
conv["cv"] = conv["std"] / conv["mean"]          # 变异系数
conv["p90_p10"] = conv["p90"] - conv["p10"]      # 分位差
print(conv.round(3).to_string(index=False))

f, l = conv.iloc[0], conv.iloc[-1]
print()
for name, key in [("标准差", "std"), ("变异系数", "cv"), ("P90-P10分位差", "p90_p10")]:
    chg = (l[key] - f[key]) / f[key] * 100
    print(f"{name}: {f[key]:.3f} -> {l[key]:.3f}  ({chg:+.1f}%)")

# 是否单调下降
mono = (conv["std"].diff().dropna() < 0).mean()
print(f"\n标准差逐期下降的比例: {mono:.0%}")

fig, ax1 = plt.subplots(figsize=(9, 4.2))
ax1.plot(conv["year"], conv["mean"], "o-", color="#2563eb", label="mean")
ax1.fill_between(conv["year"], conv["p10"], conv["p90"], alpha=0.18,
                 color="#2563eb", label="P10-P90 band")
ax1.set_xlabel("year")
ax1.set_ylabel("life expectancy (years)", color="#2563eb")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(conv["year"], conv["cv"], "s--", color="#dc2626", label="CV (right)")
ax2.set_ylabel("coefficient of variation", color="#dc2626")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
plt.title("Level rises while dispersion shrinks")
plt.tight_layout()
plt.show()


## 5. 结构分解：增长来自国家自身改善还是人口权重变化

这是诊断分析的核心工具。人口加权预期寿命的变化可以拆成两块：组内效应（各国自己进步）和结构效应（人口占比向高寿命国家转移）。同样的总变化，如果主要来自结构效应，说明并非普遍改善，而是统计口径的加权错觉。这个方法在业务里叫 shift-share，用来拆解客单价、毛利率、留存率的变化同样有效。


In [ ]:
def shift_share(df, group_col, value_col, weight_col, t0, t1):
    """把加权均值的变化拆成 组内效应 + 结构效应 + 交叉项"""
    a = df[df["year"] == t0].set_index(group_col)
    b = df[df["year"] == t1].set_index(group_col)
    idx = a.index.intersection(b.index)
    a, b = a.loc[idx], b.loc[idx]

    w0 = a[weight_col] / a[weight_col].sum()
    w1 = b[weight_col] / b[weight_col].sum()
    v0, v1 = a[value_col], b[value_col]

    within = (w0 * (v1 - v0)).sum()          # 权重不变，指标改善
    between = ((w1 - w0) * v0).sum()         # 指标不变，权重迁移
    cross = ((w1 - w0) * (v1 - v0)).sum()    # 交叉项
    total = (w1 * v1).sum() - (w0 * v0).sum()
    return pd.Series({"总变化": total, "组内效应": within,
                      "结构效应": between, "交叉项": cross})

rows = []
for c in order:
    sub = gap[gap["continent"] == c]
    r = shift_share(sub, "country", "lifeExp", "pop", y0, y1)
    r.name = c
    rows.append(r)
r = shift_share(gap, "country", "lifeExp", "pop", y0, y1)
r.name = "全球"
rows.append(r)

decomp = pd.DataFrame(rows).round(2)
decomp["组内占比"] = (decomp["组内效应"] / decomp["总变化"] * 100).round(1)
print("人口加权预期寿命变化分解（岁）:")
print(decomp.to_string())
print()
print("结论：组内占比接近或超过100%，说明提升几乎全部来自各国自身改善，")
print("      而非人口向高寿命国家迁移带来的加权效应。")

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(decomp))
ax.bar(x - 0.2, decomp["组内效应"], 0.4, label="within", color="#2563eb")
ax.bar(x + 0.2, decomp["结构效应"], 0.4, label="between", color="#f59e0b")
ax.axhline(0, color="#334155", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(decomp.index, rotation=15)
ax.set_ylabel("contribution (years)")
ax.set_title("Shift-share decomposition")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 6. 异常识别：哪些国家出现倒退

总量向好会掩盖局部恶化。这里做两层筛查：一是用 IQR 规则找出长期改善幅度显著偏低的国家（离群点），二是逐期差分找出单期最大跌幅，定位具体年份。真实数据里这些异常都对应可查证的历史事件，这正是数据分析能落到现实的地方。


In [ ]:
first = gap[gap["year"] == y0].set_index("country")["lifeExp"]
last = gap[gap["year"] == y1].set_index("country")["lifeExp"]
delta = (last - first).dropna().sort_values()
cont = gap.drop_duplicates("country").set_index("country")["continent"]

q1, q3 = delta.quantile([0.25, 0.75])
iqr = q3 - q1
low = q1 - 1.5 * iqr
print(f"改善幅度分布: 中位数 {delta.median():.2f} 岁, IQR [{q1:.2f}, {q3:.2f}], 下界 {low:.2f} 岁")

out = delta[delta < low]
print(f"IQR 规则命中 {len(out)} 个国家")
if out.empty:
    print(f"  -> 统计规则未命中: 改善幅度本身离散度很大(IQR 宽 {iqr:.1f} 岁),")
    print(f"     下界被推到 {low:.1f} 岁这种负值, 只有极端崩溃才可能触发。")
    print("  -> 这是真实数据的常态。阈值失灵时应改用业务规则, 而不是调参数硬凑出异常。")

neg = delta[delta < 0]
print(f"\n规则A 净倒退({y0}-{y1} 寿命下降)的国家 {len(neg)} 个: {list(neg.index) if len(neg) else '无'}")

tail = delta.head(8)
print(f"\n规则B 改善垫底 8 国(对比全球中位数 {delta.median():.1f} 岁):")
print(pd.DataFrame({
    "改善幅度": tail.round(2),
    "洲": cont.reindex(tail.index),
    f"{y1}寿命": last.reindex(tail.index).round(1),
}).to_string())

# 逐期差分，定位最坏的单期跌幅
g = gap.sort_values(["country", "year"])
g["chg"] = g.groupby("country")["lifeExp"].diff()
worst = g.dropna(subset=["chg"]).nsmallest(8, "chg")[["country", "year", "lifeExp", "chg"]]
print("\n单期跌幅最大的记录（每期=5年）:")
print(worst.round(2).to_string(index=False))
print("\n这些年份可与卢旺达大屠杀、柬埔寨内战、撒哈拉以南艾滋病流行等事件对照验证。")

flagged = list(dict.fromkeys(list(neg.index) + list(tail.index)))
print(f"\n两条业务规则合并后的关注名单 {len(flagged)} 个: {flagged}")

focus = flagged[:5]
fig, ax = plt.subplots(figsize=(9, 4.4))
for c in focus:
    s = gap[gap["country"] == c]
    ax.plot(s["year"], s["lifeExp"], "o-", lw=1.8, ms=4, label=c)
ax.plot(conv["year"], conv["mean"], "k--", lw=2, label="global mean")
ax.set_xlabel("year")
ax.set_ylabel("life expectancy")
ax.set_title("Countries that fell behind")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. 相关不等于因果：收入与寿命的 Preston 曲线

找到倒退国家后，自然要问驱动因素。人均 GDP 是最直观的候选变量，但原始散点是弯曲的，直接算线性相关会低估关系强度。对收入取对数后关系近似线性，这就是经济学里的 Preston 曲线。注意最后的提醒：强相关只提示方向，不能直接当因果结论用。


In [ ]:
snap = gap[gap["year"] == y1].copy()
snap["log_gdp"] = np.log10(snap["gdpPercap"])

r_raw = snap["gdpPercap"].corr(snap["lifeExp"])
r_log = snap["log_gdp"].corr(snap["lifeExp"])
r_spearman = snap["gdpPercap"].corr(snap["lifeExp"], method="spearman")
print(f"{y1} 年 人均GDP vs 预期寿命")
print(f"  Pearson(原始)   = {r_raw:.3f}")
print(f"  Pearson(取对数) = {r_log:.3f}   <- 关系近似线性后大幅提升")
print(f"  Spearman(秩)    = {r_spearman:.3f}")

# 最小二乘拟合（numpy，无需额外依赖）
b, a = np.polyfit(snap["log_gdp"], snap["lifeExp"], 1)
snap["fitted"] = a + b * snap["log_gdp"]
snap["resid"] = snap["lifeExp"] - snap["fitted"]
r2 = 1 - (snap["resid"] ** 2).sum() / ((snap["lifeExp"] - snap["lifeExp"].mean()) ** 2).sum()
print(f"\n拟合: lifeExp = {a:.1f} + {b:.1f} * log10(gdpPercap),  R^2 = {r2:.3f}")
print(f"解读: 人均GDP 每翻 10 倍，预期寿命约增加 {b:.1f} 岁")

print("\n同等收入下寿命最低（负残差最大）:")
print(snap.nsmallest(5, "resid")[["country", "gdpPercap", "lifeExp", "resid"]].round(2).to_string(index=False))
print("\n同等收入下寿命最高（正残差最大）:")
print(snap.nlargest(5, "resid")[["country", "gdpPercap", "lifeExp", "resid"]].round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
for c in order:
    s = snap[snap["continent"] == c]
    ax.scatter(s["gdpPercap"], s["lifeExp"], s=np.sqrt(s["pop"]) / 400,
               alpha=0.65, label=c)
xs = np.linspace(snap["log_gdp"].min(), snap["log_gdp"].max(), 50)
ax.plot(10 ** xs, a + b * xs, "k--", lw=2, label="log fit")
ax.set_xscale("log")
ax.set_xlabel("GDP per capita (log scale)")
ax.set_ylabel("life expectancy")
ax.set_title(f"Preston curve, {y1} (bubble = population)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. 结论汇总：把证据压缩成一张决策表

分析的终点是可执行的结论。这一步把前面 7 步的关键数字汇总成结论表，每条结论都标注支撑证据来自哪一步、以及置信程度。养成这个习惯，汇报时就不会出现「我觉得」这类无根据的表述。


In [ ]:
summary = pd.DataFrame([
    ["全球预期寿命大幅提升",
     f"{conv.iloc[0]['mean']:.1f} -> {conv.iloc[-1]['mean']:.1f} 岁", "步骤3", "高"],
    ["国家间差距在收敛",
     f"变异系数 {conv.iloc[0]['cv']:.3f} -> {conv.iloc[-1]['cv']:.3f}，标准差逐期下降占比 {mono:.0%}", "步骤4", "高"],
    ["提升源于各国自身改善，非人口加权错觉",
     f"全球组内效应占比 {decomp.loc['全球', '组内占比']:.0f}%", "步骤5", "高"],
    ["局部存在明显倒退",
     f"{len(neg)} 国净倒退，垫底 8 国改善 <= {tail.max():.1f} 岁，"
     f"最坏单期跌幅 {worst['chg'].min():.1f} 岁", "步骤6", "高"],
    ["收入与寿命强相关但非线性",
     f"取对数后 r={r_log:.2f}，R^2={r2:.2f}", "步骤7", "中（相关非因果）"],
], columns=["结论", "关键证据", "来源", "置信度"])

print("=" * 92)
print("分析结论表")
print("=" * 92)
print(summary.to_string(index=False))

print("\n" + "=" * 92)
print("行动建议（按优先级）")
print("=" * 92)
for i, (act, why) in enumerate([
    ("资源优先投向步骤6识别出的倒退国家", "总量向好掩盖了局部恶化，边际收益最高"),
    ("对负残差国家做专项诊断", "收入已达标但寿命偏低，问题在医疗体系而非经济总量"),
    ("以正残差国家为对标样本", "同等收入下表现更优，其公共卫生政策可复制"),
    ("补充时间序列因果推断", "本次仅证明相关性，需用双重差分等方法验证政策效果"),
], 1):
    print(f"{i}. {act}\n   依据: {why}")

print("\n分析局限: 数据为 5 年间隔的国家级聚合值，无法反映国内区域差异；")
print("          1952-2007 区间不含近年疫情冲击。")


## 结论与表达

- 全球预期寿命从 1952 年的约 49 岁提升到 2007 年的约 67 岁，同时国家间差距持续收敛，标准差与变异系数同向下降。
- shift-share 分解显示提升几乎全部来自各国自身改善（组内效应），而非人口向高寿命国家迁移造成的加权错觉。
- 总量向好掩盖了局部恶化：IQR 阈值在这份数据上并未命中任何国家（改善幅度本身离散度过大），改用业务规则后才定位到净倒退国家与改善垫底群体，逐期差分进一步锁定具体年份，可与真实历史事件对照验证。
- 人均 GDP 与预期寿命呈强对数关系（Preston 曲线），收入翻 10 倍约对应寿命增加数岁；残差分析能区分「钱花在了刀刃上」和「有钱但健康产出低」两类国家。


## 项目验收清单

- 能说明为什么要先审计面板完整性再做任何聚合
- 能解释组内效应与结构效应的差别，并举一个业务场景的例子
- 能用 IQR 规则完成一次异常国家筛查并解释阈值来源
- 能说明为什么对 GDP 取对数后相关系数会提升，以及为什么不能由此得出因果结论

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 Gapminder 真实面板数据（142国 × 12期 = 1704行），完成收敛性分析、人口加权增量分解与冲击事件定位，学习诊断型分析范式。


### 你已经完成

- 读取真实公开数据集并核验面板结构的完整性
- 用离散度指标判断国家间差距是收敛还是扩大
- 用 shift-share 方法把大洲变化分解为国内改善与人口结构两部分
- 用 IQR 与逐期差分定位异常国家和冲击发生的年份
- 区分相关关系的强度与函数形式，避免把相关写成因果


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 加载 Gapminder 数据并记录来源与字段口径 |
| 步骤 2 | 完成面板结构审计与取值范围检查 |
| 步骤 3 | 对比首末年份的分布形态，识别分布是否变窄 |
| 步骤 4 | 计算标准差、变异系数与 P90-P10 差距，判定收敛方向 |
| 步骤 5 | 用 shift-share 分解各洲预期寿命变化的来源 |
| 步骤 6 | 定位倒退国家与单期最大跌幅发生的年份 |
| 步骤 7 | 分析预期寿命与人均GDP的相关形式随时间的演变 |
| 步骤 8 | 输出事实→假设→验证方案的三段式结论 |


### 质量与结论提醒

- （country, year）组合是否唯一，可否作为面板主键
- 是否为平衡面板：每个国家的观测期数是否一致
- 年份间隔是否等距，能否直接做逐期差分
- 全球预期寿命从 1952 年的约 49 岁提升到 2007 年的约 67 岁，同时国家间差距持续收敛，标准差与变异系数同向下降。
- shift-share 分解显示提升几乎全部来自各国自身改善（组内效应），而非人口向高寿命国家迁移造成的加权错觉。
- 总量向好掩盖了局部恶化：IQR 阈值在这份数据上并未命中任何国家（改善幅度本身离散度过大），改用业务规则后才定位到净倒退国家与改善垫底群体，逐期差分进一步锁定具体年份，可与真实历史事件对照验证。
- 人均 GDP 与预期寿命呈强对数关系（Preston 曲线），收入翻 10 倍约对应寿命增加数岁；残差分析能区分「钱花在了刀刃上」和「有钱但健康产出低」两类国家。


### 项目交付检查

- [ ] 能说明为什么要先审计面板完整性再做任何聚合
- [ ] 能解释组内效应与结构效应的差别，并举一个业务场景的例子
- [ ] 能用 IQR 规则完成一次异常国家筛查并解释阈值来源
- [ ] 能说明为什么对 GDP 取对数后相关系数会提升，以及为什么不能由此得出因果结论
